# Wessex Stage3B: frozen CPU development replay

This notebook reuses the fixed `stage3b_20261011_v1` protocol. It never evaluates 2024–2025, expands the GRU/source pool, trains intervals, or starts paid compute. The scientific core is locally tested; live Colab login and Drive mounting have **not** been certified. Derived QA-v3 and all available daily results are real archived experiment evidence.

Download all Stage3B ZIP parts plus outer `SHA256SUMS` from the delivery. Put them in the Drive folder configured below. Every ZIP is below24MiB. `review_cached` imports the complete evidence and verifies/resumes completed tasks; `fresh_smoke` imports only Code_and_Reproduce into a **different new project directory** and actually trains the twelve prescribed small tasks from scratch. Never change frozen data/configuration to make an old cache appear compatible. Stage3A arithmetic is reproducible from the full predictions category; its complete training replay has separately documented historical dependencies.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, datetime
USE_GOOGLE_DRIVE = True
REPLAY_MODE = "review_cached"  # or "fresh_smoke"; select before import
RUN_FROZEN_FULL = False        # only set True to complete the same frozen CPU round
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TASK_DRIVE_ROOT = Path('/content/drive/MyDrive/WessexStage3B')
else:
    TASK_DRIVE_ROOT = Path('/content/WessexStage3B')
PACKAGE_DIR = TASK_DRIVE_ROOT / 'packages'
PROJECT_ROOT = TASK_DRIVE_ROOT / ('review_project' if REPLAY_MODE == 'review_cached' else 'fresh_smoke_project')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print('Place delivery ZIPs and SHA256SUMS in:', PACKAGE_DIR)
print('Persistent project:', PROJECT_ROOT)
if REPLAY_MODE not in {'review_cached', 'fresh_smoke'}:
    raise ValueError('Unknown mode')


## Verify and import

The bootstrap below is the bundled importer. It checks archive size, ZIP CRC, all member SHA256 values, internal manifests, outer checksums, path safety and conflicting existing files. Large archived CSVs are restored only after compressed and original hashes agree. Same-content repeat imports are allowed; different files are never overwritten.


In [ ]:
"""Safely verify and merge independently extractable Stage3B delivery ZIPs.

Existing different files, traversal, symlinks, unmanifested members, duplicates,
oversize ZIPs and incomplete SHA verification are rejected before import.
"""
from __future__ import annotations
import argparse
import gzip
import hashlib
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
import zipfile

LIMIT = 24*1024**2


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for data in iter(lambda: f.read(1024**2), b''):
            h.update(data)
    return h.hexdigest()


def safe_path(name):
    p = PurePosixPath(name)
    if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or name.endswith('/'):
        raise ValueError(f'Unsafe member: {name}')
    return p


def import_archives(archives, destination, outer_checks=None):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    outer = {}
    if outer_checks is not None:
        for line in Path(outer_checks).read_text().splitlines():
            h, name = line.split('  ', 1)
            outer[name] = h
    merged, manifests = {}, []
    with tempfile.TemporaryDirectory(prefix='stage3b_import_', dir=destination.parent) as temporary:
        staging = Path(temporary)
        for filename in map(Path, archives):
            if filename.stat().st_size >= LIMIT:
                raise ValueError('A ZIP reaches/exceeds the 24 MiB limit')
            if outer_checks is not None and outer.get(filename.name) != digest(filename):
                raise ValueError(f'Outer checksum mismatch: {filename.name}')
            with zipfile.ZipFile(filename) as z:
                names = z.namelist()
                if len(names) != len(set(names)) or z.testzip() is not None:
                    raise ValueError('Duplicate names or CRC failure')
                for info in z.infolist():
                    safe_path(info.filename)
                    if (info.external_attr >> 16) & 0o170000 == 0o120000:
                        raise ValueError('Symlinks are forbidden')
                manifest_names = [n for n in names if n.startswith('.stage3b_package_manifests/') and n.endswith('/MANIFEST.json')]
                if len(manifest_names) != 1:
                    raise ValueError('Exactly one package manifest is required')
                manifest = json.loads(z.read(manifest_names[0]))
                required = set(manifest['files_sha256']) | {manifest_names[0], manifest_names[0].replace('MANIFEST.json', 'SHA256SUMS')}
                if required != set(names):
                    raise ValueError('Unmanifested or missing ZIP members')
                internal = {}
                for line in z.read(manifest_names[0].replace('MANIFEST.json', 'SHA256SUMS')).decode().splitlines():
                    expected, name = line.split('  ', 1)
                    internal[name] = expected
                if set(internal) != set(manifest['files_sha256']) | {manifest_names[0]}:
                    raise ValueError('Internal checksum list is incomplete')
                if internal[manifest_names[0]] != hashlib.sha256(z.read(manifest_names[0])).hexdigest():
                    raise ValueError('Internal manifest SHA mismatch')
                if any(internal[name] != expected for name, expected in manifest['files_sha256'].items()):
                    raise ValueError('Internal checksum manifest disagrees with MANIFEST.json')
                for name, expected in manifest['files_sha256'].items():
                    source = staging / safe_path(name)
                    source.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(name) as f, source.open('wb') as target:
                        shutil.copyfileobj(f, target)
                    if digest(source) != expected:
                        raise ValueError(f'Member SHA mismatch: {name}')
                    if name in merged and merged[name] != expected:
                        raise ValueError(f'Conflicting bundle contents: {name}')
                    merged[name] = expected
                for name in required-set(manifest['files_sha256']):
                    source = staging/safe_path(name)
                    source.parent.mkdir(parents=True, exist_ok=True)
                    source.write_bytes(z.read(name))
                    merged[name] = digest(source)
                manifests.append(manifest)
        # Materialize large CSVs into their original paths after gzip hash verification.
        for manifest in manifests:
            for name, record in manifest.get('gzip_originals', {}).items():
                if not 0 <= record['original_bytes'] <= 512*1024**2:
                    raise ValueError('Unexpected decompressed artifact size')
                original = record['original_path']
                source = staging/safe_path(name)
                target = staging/safe_path(original)
                target.parent.mkdir(parents=True, exist_ok=True)
                with gzip.open(source, 'rb') as f, target.open('wb') as output:
                    written = 0
                    for chunk in iter(lambda: f.read(1024**2), b''):
                        written += len(chunk)
                        if written > record['original_bytes']:
                            raise ValueError('Compressed payload exceeds declared size')
                        output.write(chunk)
                if target.stat().st_size != record['original_bytes'] or digest(target) != record['original_sha256']:
                    raise ValueError(f'Gzip original SHA mismatch: {original}')
                if original in merged and merged[original] != record['original_sha256']:
                    raise ValueError('Conflicting compressed/uncompressed originals')
                merged[original] = record['original_sha256']
        # Verify all conflicts first; only then install verified payloads.
        for name, expected in merged.items():
            target = destination/safe_path(name)
            if target.is_symlink() or not target.resolve().is_relative_to(destination):
                raise ValueError('Destination escapes the import directory')
            if target.exists() and (not target.is_file() or digest(target) != expected):
                raise FileExistsError(f'Refusing to overwrite different file: {target}')
        for name in merged:
            target = destination/safe_path(name)
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(staging/safe_path(name), target)
    return dict(status='passed', archives=len(archives), verified_files=len(merged),
                crc_and_all_member_sha256_verified=True, destination=str(destination))



outer = PACKAGE_DIR / 'SHA256SUMS'
if not outer.is_file():
    raise FileNotFoundError('Copy the outer SHA256SUMS into the package folder first')
expected_names = [line.split('  ',1)[1] for line in outer.read_text().splitlines() if line.strip()]
if REPLAY_MODE == 'review_cached':
    chosen_names = expected_names
else:
    chosen_names = [name for name in expected_names if 'Code_and_Reproduce' in name]
if not chosen_names:
    raise ValueError('Required code package is missing from the checksum list')
missing = [name for name in chosen_names if not (PACKAGE_DIR/name).is_file()]
if missing:
    raise FileNotFoundError('Missing ZIP parts: '+str(missing))
receipt = import_archives([PACKAGE_DIR/name for name in chosen_names], PROJECT_ROOT, outer)
print(json.dumps(receipt, indent=2))


## Minimal CPU environment

Only the pinned numerical/plotting dependencies needed by this Ridge round are installed. PyTorch/GPU is not needed for Stage3B or archived Stage3A metric arithmetic. If separately replaying historical GRU training, follow OMITTED_DEPENDENCIES.json and install the documented CPU PyTorch version in that separate project.


In [ ]:
ENV_ROOT = Path('/content/wessex_stage3b_cpu_env')
if not (ENV_ROOT/'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ENV_ROOT)], check=True)
PYTHON = str(ENV_ROOT/'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')], check=True)
CPU_ENV = dict(os.environ, PYTHONPATH=str(PROJECT_ROOT/'src'), OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
LOG_DIR = PROJECT_ROOT/'notebook_logs'
LOG_DIR.mkdir(exist_ok=True)
def run_logged(arguments, label):
    path = LOG_DIR/(label+'.log')
    with path.open('a') as log:
        log.write('\nSTART '+datetime.datetime.now(datetime.timezone.utc).isoformat()+' '+repr(arguments)+'\n')
        log.flush()
        result = subprocess.run([PYTHON]+list(arguments), cwd=PROJECT_ROOT, env=CPU_ENV, stdout=log, stderr=subprocess.STDOUT)
    print(path.read_text()[-6000:])
    if result.returncode:
        raise RuntimeError(f'Command failed with {result.returncode}; preserved log: {path}')


In [ ]:
CHECK_CODE = """
import json
from pathlib import Path
from wastewater.stage3b_experiment import validate,execution_signature
root=Path.cwd()
config_path='configs/stage3b_20261011_v1/full.json'
cfg=json.loads((root/config_path).read_text())
validate(cfg,root)
signature,provenance=execution_signature(cfg,config_path)
print(json.dumps(dict(status='validated',signature=signature,provenance=provenance),indent=2))
"""
run_logged(['-c', CHECK_CODE], '01_config_data_code_validation')
run_logged(['-m','unittest','discover','-s','tests','-p','test_stage3b*.py','-v'], '02_software_tests')


## Twelve-condition smoke check and exact frozen continuation

The two prescribed smoke SiteIDs are13013 and13015, first2022 window, all three budgets and both horizons. Completed source/task fingerprints are checked before cached reuse; failed/incomplete task logs are retained. Fresh mode trains these tasks from the included QA data. Review mode verifies the complete imported cache. `RUN_FROZEN_FULL=True` continues only the fixed300-condition design, including the history-audited October2023 window.


In [ ]:
CONFIG = 'configs/stage3b_20261011_v1/full.json'
run_logged(['-m','wastewater.stage3b_experiment','--config',CONFIG,'--workers','2','--smoke'], '03_smoke_or_cached_check')
if REPLAY_MODE == 'review_cached' or RUN_FROZEN_FULL:
    run_logged(['-m','wastewater.stage3b_experiment','--config',CONFIG,'--workers','2'], '04_frozen_full_or_resume')
else:
    print('Fresh twelve-condition smoke finished. Full300-condition experiment was not requested in this notebook session.')


## Recalculate actual metrics

Archived GRU seeds are averaged as losses per date, never combined into an undeclared prediction ensemble. Recalculation makes no claim to have independently retrained all historical models. In fresh-smoke mode these full-data checks are skipped unless the complete frozen experiment has actually finished.


In [ ]:
if REPLAY_MODE == 'review_cached':
    run_logged(['scripts/recalculate_stage3a_delivery.py','--output','outputs/recomputed_stage3a'], '05_stage3a_independent_arithmetic')
if REPLAY_MODE == 'review_cached' or RUN_FROZEN_FULL:
    run_logged(['scripts/summarize_stage3b.py','--config',CONFIG,
                '--predictions','reports/stage3b_20261011_v1/results/predictions_all.csv.gz',
                '--output','outputs/recomputed_stage3b'], '06_stage3b_metric_recalculation')
    manifest = json.loads((PROJECT_ROOT/'runs/stage3b_20261011_v1/full/manifest.json').read_text())
    print(json.dumps({key:manifest.get(key) for key in ['status','expected_conditions','completed_conditions','failed_conditions','not_executed_conditions','counts_this_call']}, indent=2))


## Export a checked record of this notebook session

Drive project files and failures persist. The final ZIP below contains session logs, recalculated tables and a truthful completion note; it excludes raw data and training checkpoints. It is split if needed so every exported ZIP remains below24MiB. Downloading requires an active browser session. This does not upload/publicize results or submit a paper.


In [ ]:
import zipfile
EXPORT_DIR = TASK_DRIVE_ROOT/'session_exports'
EXPORT_DIR.mkdir(exist_ok=True)
manifest_path = PROJECT_ROOT/'runs/stage3b_20261011_v1/full/manifest.json'
session = dict(mode=REPLAY_MODE,run_frozen_full_requested=RUN_FROZEN_FULL,
               session_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
               completed_full_manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else None,
               live_drive_mounted_this_session=USE_GOOGLE_DRIVE,
               final_period_performance_not_enabled=True)
(LOG_DIR/'SESSION_STATUS.json').write_text(json.dumps(session,indent=2)+'\n')
files = sorted([p for folder in [LOG_DIR,PROJECT_ROOT/'outputs'] if folder.exists() for p in folder.rglob('*') if p.is_file()])
chunks, group, size = [], [], 0
for path in files:
    if path.stat().st_size >= 22*1024**2:
        raise ValueError('One exported artifact is too large; preserve it in Drive rather than dropping rows')
    if group and size+path.stat().st_size > 22*1024**2:
        chunks.append(group); group=[]; size=0
    group.append(path); size+=path.stat().st_size
if group: chunks.append(group)
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
checks=[]
for index,group in enumerate(chunks,1):
    destination=EXPORT_DIR/f'Wessex_Stage3B_session_{stamp}_Part{index:02d}.zip'
    members={str(path.relative_to(PROJECT_ROOT)):digest(path) for path in group}
    with zipfile.ZipFile(destination,'x',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for path in group:z.write(path,str(path.relative_to(PROJECT_ROOT)))
        z.writestr('SHA256SUMS',''.join(f'{h}  {name}\n' for name,h in sorted(members.items())))
    assert destination.stat().st_size <24*1024**2
    with zipfile.ZipFile(destination) as z:
        assert z.testzip() is None
        for name,expected in members.items():assert hashlib.sha256(z.read(name)).hexdigest()==expected
    checks.append(dict(path=str(destination),bytes=destination.stat().st_size,sha256=digest(destination)))
print(json.dumps(checks,indent=2))
print('STOP. Do not broaden the experiment based on these scores.')
